In [1]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)

In [2]:
import pandas as pd

In [4]:
# load data
DATA_PATH = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"

df = pd.read_stata(
    DATA_PATH / "Pop_Data_Appended_NoPopcorn.dta",
    convert_categoricals=False,
 )

/tmp/ipykernel_17270/2830712342.py:5: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(


In [5]:
df.head()

,ID,year,HHSize6,NOFAd3,NOFCh3,addnum,Sex,ag16g10,Age16g5,Age35g,...,InterviewDuration1,InterviewDuration2,InterviewDuration3,InterviewDuration4,ReloadSurvey,QCreCAPTCHA,SurveyWeight,AgeWeight,QCResults,sat
0,2700001.0,2017.0,2.0,2.0,0.0,13.0,1.0,6.0,13.0,18.0,...,,,,,,,,,,9.0
1,2700002.0,2017.0,2.0,2.0,0.0,15.0,2.0,2.0,4.0,9.0,...,,,,,,,,,,7.0
2,2700003.0,2017.0,4.0,2.0,2.0,18.0,2.0,-1.0,-1.0,3.0,...,,,,,,,,,,NaN
3,2700005.0,2017.0,3.0,1.0,2.0,9.0,2.0,-1.0,-1.0,5.0,...,,,,,,,,,,NaN
4,2700006.0,2017.0,4.0,2.0,2.0,7.0,1.0,3.0,7.0,12.0,...,,,,,,,,,,9.0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 57425 entries, 0 to 57424
Columns: 4217 entries, ID to sat
dtypes: datetime64[s](6), float32(93), float64(3956), str(162)
memory usage: 1.8 GB


In [9]:
print(f"shape: {df.shape}")
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"text columns: {len(text_columns)}")
text_columns[:10]

df[text_columns[:5]].head() if text_columns else "No text columns found"

shape: (57425, 4217)
text columns: 162


,startlanguage,seed,token,ipaddr,refurl
0,,,,,
1,,,,,
2,,,,,
3,,,,,
4,,,,,


In [11]:
df['year'].isnull().sum()

np.int64(47443)

In [13]:
datetime_columns = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
print(f"datetime columns: {len(datetime_columns)}")
datetime_columns

(
    df[datetime_columns]
    .agg(["min", "max"])
    .T
    .sort_index()
    if datetime_columns
    else "No datetime columns found"
 )

datetime columns: 6


,min,max
Age2,1936-01-01,2006-02-01
BornDate,1924-01-01,2006-02-01
datestamp,2023-04-06,2024-03-14
sedTime,1970-01-01,1970-01-01
startdate,2023-04-06,2024-03-14
submitdate,2023-04-06,2024-04-10


In [15]:
(
    df[datetime_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_values")
    .to_frame()
    if datetime_columns
    else "No datetime columns found"
 )

,missing_values
Age2,52803
sedTime,52770
BornDate,45272
submitdate,40287
startdate,39653
datestamp,39653


In [16]:
(
    (df[datetime_columns].isna().mean() * 100)
    .sort_values(ascending=False)
    .round(2)
    .rename("missing_pct")
    .to_frame()
    if datetime_columns
    else "No datetime columns found"
 )

,missing_pct
Age2,91.95
sedTime,91.89
BornDate,78.84
submitdate,70.16
startdate,69.05
datestamp,69.05


In [17]:
dropped_datetime_columns = datetime_columns.copy()
df = df.drop(columns=dropped_datetime_columns)
print(f"Dropped {len(dropped_datetime_columns)} datetime columns")
print(f"New shape: {df.shape}")
dropped_datetime_columns

Dropped 6 datetime columns
New shape: (57425, 4211)


['submitdate', 'startdate', 'datestamp', 'BornDate', 'sedTime', 'Age2']

In [18]:
float32_columns = df.select_dtypes(include=["float32"]).columns.tolist()
print(f"float32 columns: {len(float32_columns)}")
float32_columns[:50]

float32 columns: 93


['omdiast',
 'omsyst',
 'ommap',
 'ompuls',
 'MO5L',
 'SC5L',
 'UA5L',
 'PD5L',
 'AD5L',
 'EQvas',
 'SAH',
 'sex',
 'age',
 'age10',
 'educ',
 'inco3',
 'inco5',
 'empl',
 'hpempl',
 'mars',
 'size',
 'limc',
 'scls',
 'urbn',
 'help',
 'util_rowen',
 'FULLHEALTH',
 'anyprob',
 'mo2cat',
 'sc2cat',
 'ua2cat',
 'pd2cat',
 'ad2cat',
 'LSS',
 'LSS_rs',
 'weight_clean',
 'height_meters',
 'bmi_calc',
 'bmi_cat',
 'disut_mo',
 'disut_sc',
 'disut_ua',
 'disut_pd',
 'disut_ad',
 'disut_total',
 'EQ_index',
 'age7cat',
 'srh',
 'smoke_ever',
 'smoke_ecig']

In [19]:
(
    pd.DataFrame({
        "missing_values": df[float32_columns].isna().sum(),
        "missing_pct": (df[float32_columns].isna().mean() * 100).round(2),
    })
    .sort_values(["missing_pct", "missing_values"], ascending=False)
    if float32_columns
    else "No float32 columns found"
 )

,missing_values,missing_pct
st_part,57419,99.99
lb_part,57419,99.99
ft_part,57271,99.73
in_part,57271,99.73
height_m,56654,98.66
...,...,...
util_rowen,19421,33.82
smoke_ecig,18743,32.64
srh,11211,19.52
age7cat,3976,6.92


In [20]:
missing_pct = df[float32_columns].isna().mean().mul(100)
bucket_labels = [
    "0-10",
    "11-20",
    "21-30",
    "31-40",
    "41-50",
    "51-60",
    "61-70",
    "71-80",
    "81-90",
    "91-100",
]

(
    pd.DataFrame({
        "column": missing_pct.index,
        "missing_pct": missing_pct.values,
        "bucket": pd.cut(
            missing_pct,
            bins=[-0.01, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
            labels=bucket_labels,
        ),
    })
    .groupby("bucket", observed=False)["column"]
    .agg(list)
    .to_frame("columns")
    .loc[bucket_labels]
    if float32_columns
    else "No float32 columns found"
 )

,columns
bucket,
0-10,"[age7cat, dataset]"
11-20,[srh]
21-30,[]
31-40,"[util_rowen, smoke_ecig, resp]"
41-50,"[weight_clean, height_meters, bmi_calc, bmi_ca..."
51-60,"[alcohol_yr, mus]"
61-70,"[omdiast, omsyst, ommap, ompuls, MO5L, SC5L, U..."
71-80,[]
81-90,"[age, age10, educ, inco3, inco5, empl, hpempl,..."


In [22]:
bucketed_float32_columns = (
    pd.DataFrame({
        "column": missing_pct.index,
        "missing_pct": missing_pct.values,
        "bucket": pd.cut(
            missing_pct,
            bins=[-0.01, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
            labels=bucket_labels,
        ),
    })
    .groupby("bucket", observed=False)["column"]
    .agg(list)
    .reindex(bucket_labels, fill_value=[])
    if float32_columns
    else None
)

if bucketed_float32_columns is None:
    print("No float32 columns found")
else:
    for bucket, columns in bucketed_float32_columns.items():
        print(f"\n{bucket} ({len(columns)} columns)")
        for column in columns:
            print(column)


0-10 (2 columns)
age7cat
dataset

11-20 (1 columns)
srh

21-30 (0 columns)

31-40 (3 columns)
util_rowen
smoke_ecig
resp

41-50 (6 columns)
weight_clean
height_meters
bmi_calc
bmi_cat
skin
sat

51-60 (2 columns)
alcohol_yr
mus

61-70 (30 columns)
omdiast
omsyst
ommap
ompuls
MO5L
SC5L
UA5L
PD5L
AD5L
EQvas
SAH
sex
FULLHEALTH
anyprob
mo2cat
sc2cat
ua2cat
pd2cat
ad2cat
LSS
LSS_rs
disut_mo
disut_sc
disut_ua
disut_pd
disut_ad
disut_total
EQ_index
smoke_ever
obese

71-80 (0 columns)

81-90 (41 columns)
age
age10
educ
inco3
inco5
empl
hpempl
mars
size
limc
scls
urbn
help
diabetes
bowel
nberpt7_19
sberpt7_19
wgl250ml_19
wgl175ml_19
wgl125ml_19
SBeerQ19a
d7popu_19
CNBeerQa
hchrs1
NBeerQ22a
SBeerQ22a
NCid22a
SCid22a
nberpt7_22
sberpt7_22
ncidpt7_22
scidpt7_22
wgl250ml_22
wgl175ml_22
d7stcidu_22
d7popu_22
CWL7Bt
CNBeerQ22a
CNCid22a
CSCid22a
WtChAdN

91-100 (8 columns)
st_part
lb_part
w_num
weight_kg
h_num
ft_part
in_part
height_m


In [23]:
high_missing_float32_columns = missing_pct[missing_pct > 90].index.tolist()
df = df.drop(columns=high_missing_float32_columns)
float32_columns = df.select_dtypes(include=["float32"]).columns.tolist()
print(f"Dropped {len(high_missing_float32_columns)} float32 columns with >90% missingness")
print(f"New shape: {df.shape}")
high_missing_float32_columns

Dropped 8 float32 columns with >90% missingness
New shape: (57425, 4203)


['st_part',
 'lb_part',
 'w_num',
 'weight_kg',
 'h_num',
 'ft_part',
 'in_part',
 'height_m']